## 1. Install Required Packages

In [ ]:
!pip install rioxarray rasterio tqdm shapely requests

## 2. Import Libraries

In [ ]:
import rioxarray
import os
import requests
from tqdm import tqdm
from zipfile import ZipFile
import shutil

## 3. Set USA Bounding Box and Paths

In [ ]:
# USA bounding box (includes Alaska and Hawaii)
LAT_MIN, LAT_MAX = 18.91619, 71.3577635769
LON_MIN, LON_MAX = -171.791110603, -66.96466

# Set your output path
ROOT_DIR = 'USA_WorldClim_2.5m/'
os.makedirs(ROOT_DIR, exist_ok=True)

# Data types to download
DATA_TYPES = ['tmin', 'tmax', 'prec']

print(f"Data will be saved to: {ROOT_DIR}")
print(f"USA Bounding Box: Lat [{LAT_MIN}, {LAT_MAX}], Lon [{LON_MIN}, {LON_MAX}]")

## 4. Download and Extract Functions

In [ ]:
def download_file(url, target_folder):
    """Download a file with progress bar"""
    local_filename = os.path.join(target_folder, url.split('/')[-1])
    
    # Skip if already downloaded
    if os.path.exists(local_filename):
        print(f"Already exists: {local_filename}")
        return local_filename
    
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            total_size = int(r.headers.get('content-length', 0))
            
            with open(local_filename, 'wb') as f:
                with tqdm(total=total_size, unit='B', unit_scale=True, desc=local_filename) as pbar:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))
            
            print(f"Downloaded: {local_filename}")
            return local_filename
    except Exception as e:
        print(f"Failed to download {url}: {e}")
        return None


def unzip_file(zip_path, extract_to):
    """Unzip a file and remove the zip"""
    try:
        with ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"Extracted: {zip_path}")
        os.remove(zip_path)
        print(f"Removed: {zip_path}")
    except Exception as e:
        print(f"Failed to unzip {zip_path}: {e}")


def organize_files_by_year(data_type_dir, decade_folder):
    """Organize files into year subdirectories"""
    decade_path = os.path.join(data_type_dir, decade_folder)
    
    if not os.path.isdir(decade_path):
        return
    
    print(f"Organizing files in {decade_path}...")
    
    for file_name in tqdm(os.listdir(decade_path), desc=f"Organizing {decade_folder}"):
        if file_name.endswith('.tif'):
            # Extract year from filename (e.g., wc2.1_2.5m_tmin_1960-01.tif -> 1960)
            year = file_name.split('_')[-1].split('-')[0]
            
            # Create year folder
            year_folder = os.path.join(decade_path, year)
            os.makedirs(year_folder, exist_ok=True)
            
            # Move file
            src_file = os.path.join(decade_path, file_name)
            dest_file = os.path.join(year_folder, file_name)
            shutil.move(src_file, dest_file)

## 5. Download WorldClim Historical Monthly Data

Downloads data from 1960-2018 organized by decades

In [ ]:
# WorldClim 2.1 base URL for 2.5 arcmin resolution
BASE_URL = "https://geodata.ucdavis.edu/climate/worldclim/2_1/base/"

# Year blocks for historical data
YEAR_BLOCKS = [
    "1960-1969", "1970-1979", "1980-1989", "1990-1999",
    "2000-2009", "2010-2018"
]

print(f"{'='*70}")
print(f"Downloading WorldClim 2.1 Historical Monthly Data for USA")
print(f"Resolution: 2.5 arcmin (~5km)")
print(f"Period: 1960-2018")
print(f"Variables: {', '.join(DATA_TYPES)}")
print(f"{'='*70}\n")

for data_type in DATA_TYPES:
    print(f"\n{'='*70}")
    print(f"Processing: {data_type.upper()}")
    print(f"{'='*70}\n")
    
    # Create directory for this data type
    data_type_dir = os.path.join(ROOT_DIR, data_type)
    os.makedirs(data_type_dir, exist_ok=True)
    
    # Download each year block
    for year_block in YEAR_BLOCKS:
        print(f"\n--- Downloading: {data_type} {year_block} ---")
        
        # Construct URL (e.g., wc2.1_2.5m_tmin_1960-1969.zip)
        filename = f"wc2.1_2.5m_{data_type}_{year_block}.zip"
        url = BASE_URL + filename
        
        # Download
        zip_path = download_file(url, data_type_dir)
        
        if zip_path:
            # Extract to decade folder
            decade_folder = os.path.join(data_type_dir, year_block)
            os.makedirs(decade_folder, exist_ok=True)
            unzip_file(zip_path, decade_folder)
            
            # Organize into year subfolders
            organize_files_by_year(data_type_dir, year_block)

print(f"\n{'='*70}")
print("Download complete!")
print(f"{'='*70}")

## 6. Clip to USA Bounding Box (Optional)

Clips downloaded global tiles to USA region to reduce file sizes

In [ ]:
def clip_raster_to_bbox(input_file, output_file, bbox):
    """Clip a raster file to bounding box"""
    try:
        # Read and clip
        data = rioxarray.open_rasterio(input_file, masked=True)
        clipped = data.rio.clip_box(
            minx=bbox['lon_min'],
            miny=bbox['lat_min'],
            maxx=bbox['lon_max'],
            maxy=bbox['lat_max']
        )
        
        # Save clipped version
        clipped.rio.to_raster(output_file, compress='LZW')
        return True
    except Exception as e:
        print(f"Error clipping {input_file}: {e}")
        return False


# Clip all downloaded files
USA_BBOX = {
    'lat_min': LAT_MIN,
    'lat_max': LAT_MAX,
    'lon_min': LON_MIN,
    'lon_max': LON_MAX
}

CLIPPED_DIR = 'USA_WorldClim_2.5m_Clipped/'
os.makedirs(CLIPPED_DIR, exist_ok=True)

print(f"Clipping rasters to USA bounding box...\n")

for data_type in DATA_TYPES:
    data_type_dir = os.path.join(ROOT_DIR, data_type)
    clipped_data_dir = os.path.join(CLIPPED_DIR, data_type)
    
    # Walk through all subdirectories
    for root, dirs, files in os.walk(data_type_dir):
        for file in files:
            if file.endswith('.tif'):
                input_path = os.path.join(root, file)
                
                # Recreate directory structure
                rel_path = os.path.relpath(root, data_type_dir)
                output_dir = os.path.join(clipped_data_dir, rel_path)
                os.makedirs(output_dir, exist_ok=True)
                
                output_path = os.path.join(output_dir, file)
                
                # Skip if already clipped
                if os.path.exists(output_path):
                    continue
                
                print(f"Clipping: {file}")
                clip_raster_to_bbox(input_path, output_path, USA_BBOX)

print(f"\nClipped files saved to: {CLIPPED_DIR}")

## 7. Verification

In [ ]:
# Verify files are downloaded
def check_downloaded_files():
    """Check what files were successfully downloaded"""
    
    if not os.path.exists(ROOT_DIR):
        print(f"Directory does not exist: {ROOT_DIR}")
        return
    
    print(f"Checking directory: {ROOT_DIR}\n")
    
    for data_type in DATA_TYPES:
        data_type_dir = os.path.join(ROOT_DIR, data_type)
        if os.path.exists(data_type_dir):
            print(f"\n{'='*60}")
            print(f"{data_type.upper()}:")
            print(f"{'='*60}")
            
            for year_block in sorted(os.listdir(data_type_dir)):
                year_block_path = os.path.join(data_type_dir, year_block)
                if os.path.isdir(year_block_path):
                    print(f"\n  📁 {year_block}/")
                    
                    for year in sorted(os.listdir(year_block_path)):
                        year_path = os.path.join(year_block_path, year)
                        if os.path.isdir(year_path):
                            file_count = len([f for f in os.listdir(year_path) if f.endswith('.tif')])
                            print(f"    📁 {year}/ - {file_count} .tif files")
        else:
            print(f"{data_type}: Not downloaded yet")
    
    # Total count
    print(f"\n{'='*60}")
    total_files = 0
    for data_type in DATA_TYPES:
        data_type_dir = os.path.join(ROOT_DIR, data_type)
        if os.path.exists(data_type_dir):
            count = sum([len([f for f in files if f.endswith('.tif')]) 
                        for root, dirs, files in os.walk(data_type_dir)])
            total_files += count
            print(f"{data_type}: {count} files")
    
    print(f"\nTotal .tif files: {total_files}")
    print(f"Expected: ~2,124 files (59 years × 12 months × 3 variables)")
    print(f"{'='*60}")

check_downloaded_files()